# Colab — 100k monthly (HF progress resume)

Runbook: `Fetcher/docs/COLAB_20K_RUN_RU.md`

**Модель:**
- **7 дней** × 1 сессия discover в день (до исчерпания квоты ключей) → ~100k видео (snapshot_0)
- Прогресс **всегда** в HF repo `{prefix}/dataset_100k_monthly` → `state/progress/*`
- При каждом запуске Colab: **pull прогресса** → resume с checkpoint / seen_ids / очередей
- Follow-up снапшоты (7/14/21/28d): `snapshot-poll` после недели discover

**Multi-Colab:** B/C — `Colab_20k_BC_worker.ipynb` (download shard 1/2, enrich shard 2/2)

`output_dir` можно на `/content/...` — данные восстанавливаются с HF.

## 0. CONFIG

In [ ]:
CONFIG = {
    "repo_url": "https://github.com/lebedev-ilia/TrendFlowML.git",
    "fetcher": "/content/TrendFlowML/Fetcher",
    "drive_secrets": "/content/drive/MyDrive/trendflow_secrets",
    "hf_repo_prefix": "Ilialebedev",
    "campaign_profile": "100k-monthly",
    "discover_week_days": 7,
    "interval_sec": 120,
    "metrics_discover": 9095,
    "metrics_workers": 9096,
    "parallel_colab_count": 3,
    "worker_shard_index": 0,
    "worker_shard_count": 3,
    "worker_id": "colab-100k-main",
}
CONFIG["output_dir"] = "/content/dataset_runs/100k-monthly"

import os
from pathlib import Path
OUTPUT = Path(CONFIG["output_dir"])
OUTPUT.mkdir(parents=True, exist_ok=True)
print("output_dir:", OUTPUT)
print("HF progress repo:", CONFIG["hf_repo_prefix"] + "/dataset_100k_monthly")

## 1. Drive + git + pip

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import subprocess
from pathlib import Path

dest = Path("/content/TrendFlowML")
if (dest / ".git").exists():
    subprocess.run(["git", "-C", str(dest), "pull"], check=False)
else:
    subprocess.run(["git", "clone", CONFIG["repo_url"], str(dest)], check=True)

subprocess.run(["pip", "install", "-q", "-r", str(Path(CONFIG["fetcher"]) / "requirements.txt")], check=True)
subprocess.run(["pip", "install", "-q", "-U", "huggingface_hub", "yt-dlp", "pytubefix"], check=True)
print("ready")

## 2. HF_TOKEN + secrets

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN").strip()
except Exception:
    pass

if not os.environ.get("HF_TOKEN", "").startswith("hf_"):
    raise RuntimeError("Set Colab Secret HF_TOKEN or os.environ[\"HF_TOKEN\"]")

hf_path = Path(CONFIG["output_dir"]) / ".hf_token"
hf_path.write_text(os.environ["HF_TOKEN"].strip() + "\n", encoding="utf-8")
print("HF_TOKEN OK, saved for subprocess:", hf_path)

## 3. Discover (день N из 7)

Без `--limit`: собирает до квоты. При старте **скачивает прогресс с HF** (категории, checkpoint, seen_ids).
Запускайте **каждый день** новую сессию Colab (до 7 дней).

In [ ]:
import subprocess, os
from pathlib import Path

FETCHER = Path(CONFIG["fetcher"])
cmd = [
    "python", "scripts/colab_20k_bootstrap.py",
    "--campaign-profile", CONFIG["campaign_profile"],
    "--role", "discover",
    "--output-dir", CONFIG["output_dir"],
    "--hf-repo-prefix", CONFIG["hf_repo_prefix"],
    "--parallel-colab-count", str(CONFIG["parallel_colab_count"]),
    "--metrics-port", str(CONFIG["metrics_discover"]),
]
log = Path("/content/discover_100k_log.txt")
with open(log, "w") as fh:
    p = subprocess.Popen(cmd, cwd=str(FETCHER), stdout=fh, stderr=subprocess.STDOUT, env=os.environ.copy())
print("discover pid", p.pid, "log:", log)
print("tail: !tail -f", log)

## 4. Workers (download + enrich + HF upload)

In [ ]:
import subprocess, os
from pathlib import Path

cmd = [
    "python", "scripts/colab_20k_bootstrap.py",
    "--campaign-profile", CONFIG["campaign_profile"],
    "--role", "workers",
    "--output-dir", CONFIG["output_dir"],
    "--hf-repo-prefix", CONFIG["hf_repo_prefix"],
    "--interval", str(CONFIG["interval_sec"]),
    "--metrics-port", str(CONFIG["metrics_workers"]),
    "--worker-id", CONFIG["worker_id"],
    "--worker-shard-index", str(CONFIG["worker_shard_index"]),
    "--worker-shard-count", str(CONFIG["worker_shard_count"]),
    "--parallel-colab-count", str(CONFIG["parallel_colab_count"]),
    "--hf-coord",
]
log = Path("/content/workers_100k_log.txt")
with open(log, "w") as fh:
    p = subprocess.Popen(cmd, cwd=str(Path(CONFIG["fetcher"])), stdout=fh, stderr=subprocess.STDOUT, env=os.environ.copy())
print("workers pid", p.pid)

## 5. Snapshot-poll (после 7 дней discover)

Follow-up снапшоты по `snapshot_schedule_days: [0,7,14,21,28]`. Прогресс `video_schedule` / `snapshot_completion` тоже на HF.

In [ ]:
import subprocess, os
from pathlib import Path

cmd = [
    "python", "scripts/colab_20k_bootstrap.py",
    "--campaign-profile", CONFIG["campaign_profile"],
    "--role", "snapshot-poll",
    "--output-dir", CONFIG["output_dir"],
]
raise SystemExit(subprocess.call(cmd, cwd=str(Path(CONFIG["fetcher"])), env=os.environ.copy()))

## 6. Status + audit

In [ ]:
import subprocess, os
from pathlib import Path

FETCHER = Path(CONFIG["fetcher"])
subprocess.run([
    "python", "scripts/colab_20k_bootstrap.py",
    "--campaign-profile", CONFIG["campaign_profile"],
    "--role", "status",
    "--output-dir", CONFIG["output_dir"],
], cwd=str(FETCHER), env=os.environ.copy(), check=False)

out = Path(CONFIG["output_dir"])
subprocess.run([
    "python", "scripts/audit_dataset_run.py", str(out),
    "--out", str(out / "audit.json"), "--check-hf",
], cwd=str(FETCHER), check=False)